# Analysis for the $\sin$ benchmark

The datasets considered as follows.

1. `sin-noise-0`:
    Each $x_j$ is selected uniformly at random between $-\pi$ and $\pi$, $j = 1 \dots 100$.
    Then $y_j = \sin x_j$.
2. `sin-noise-1`:
    Same process as `sin-noise-0` but $y_j = \sin x_j + \eta_j$ where $\eta_j$ has a normal distribution with mean $0$ and standard deviation $0.1$.

## Prelude

In [ ]:
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

In [ ]:
def lambda_op_plot(
    data,
    spiffy_titles=None,
    complexity_col="complexity",
    mse_col="mse",
    complexity_binwidth=10,
    mse_binwidth=0.02,
    complexity_lims=(0, 169),
    mse_lims=(6e-5, 1.1e-4),
    mse_bound=1e-4,
    picture_dir=Path("Generated/Pictures"),
    file_stem=None,
    **kwargs
):
    mse_lims_log = (np.log10(mse_lims[0]), np.log10(mse_lims[1]))
    fig = sns.displot(
        data,
        x=complexity_col,
        y=mse_col,
        col="Lopstr",
        log_scale=[False, True],
        binwidth=(complexity_binwidth, mse_binwidth),
        binrange=(complexity_lims, mse_lims_log),
    )
    # Add a horizontal line for the MSE bound in each subplot
    for ax in fig.axes.flat:
        ax.axhline(y=mse_bound, color="red", linestyle="--", label="MSE bound")
    fig.set(xlim=complexity_lims, ylim=mse_lims)
    if spiffy_titles is not None:
        for ax, title in zip(fig.axes.flat, spiffy_titles):
            ax.set_title(title)
    fig.set(**kwargs)
    au.savefig(fig, file_stem=file_stem, picture_dir=picture_dir)
    return fig

## Loading data

These are the datasets to be fit.

In [ ]:
sn0 = pd.read_csv("datasets/sin-noise-0.csv")
sn1 = pd.read_csv("datasets/sin-noise-1.csv")

In [ ]:
fig = sns.relplot(data=sn0, x="x", y="y", color="blue", aspect=1.5)
fig.set(title=r"$y = \sin x$")
au.savefig(fig, file_stem="sin-noise-0")

In [ ]:
fig = sns.relplot(data=sn1, x="x", y="y", color="blue", aspect=1.5)
fig.set(title=r"$y = \sin x + \eta$ where $\eta \sim \mathcal{N}(0, 0.1)$")
au.savefig(fig, file_stem="sin-noise-01")

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("For-analysis/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(au.parse_if_needed)
full_report["sympy_defuzz"] = full_report.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
full_report.run_set.unique()

In [ ]:
full_report.data_set.unique()

In [ ]:
full_report.sort_values(by=["run_set", "data_set", "mse"], inplace=True)

srb_key_base = "SRB-2026-06-25-1715"

srb_keys = [srb_key_base, srb_key_base + "-L4", srb_key_base + "-L2"]

Lop12 = 1.0e-12
Lop4 = 1.0e-4
Lop2 = 1.0e-2
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715", "Lop"] = Lop12
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L4", "Lop"] = Lop4
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L2", "Lop"] = Lop2

# These have to be strings because they need to be exact categorical labels for plotting
spiffy_Lopstr12 = r"$\lambda = 10^{-12}$"
spiffy_Lopstr4 = r"$\lambda = 10^{-4}$"
spiffy_Lopstr2 = r"$\lambda = 10^{-2}$"
Lopstr12 = "L0000"
Lopstr4 = "L0001"
Lopstr2 = "L0100"
full_report.loc[full_report.run_set==srb_key_base, "Lopstr"] = Lopstr12
full_report.loc[full_report.run_set==srb_key_base+"-L4", "Lopstr"] = Lopstr4
full_report.loc[full_report.run_set==srb_key_base+"-L2", "Lopstr"] = Lopstr2


In [ ]:
fr1 = full_report.copy()
fr1 = fr1.loc[fr1.data_set.isin(['sin-noise-0', 'sin-noise-1'])]

# Normalize the run set
fr1.loc[fr1.run_set==srb_key_base+"-L4", "run_set"] = srb_key_base
fr1.loc[fr1.run_set==srb_key_base+"-L2", "run_set"] = srb_key_base

fr1.set_index(["run_set", "Lopstr"], inplace=True)
fr1.sort_index(inplace=True)
srb_fr = fr1.loc[srb_key_base]

In [ ]:
fr1

In [ ]:
results_sn0 = srb_fr.loc[srb_fr.data_set == "sin-noise-0"]
results_sn1 = srb_fr.loc[srb_fr.data_set == "sin-noise-1"]

## For reference

Symbolic regression is fitting noise if it gets MSE any lower than these.

In [ ]:
sn0_mse_bound = au.mse(np.sin(sn0.x), sn0.y)
sn1_mse_bound = au.mse(np.sin(sn1.x), sn1.y)


## Analysis of results from the narrow datasets

### One period, zero noise

In [ ]:
sns.displot(data=results_sn0, x="mse", col="Lopstr", log_scale=True)

In [ ]:
sns.displot(data=results_sn0, x="complexity", col="Lopstr")

With no noise, the function $\sin x$ is recovered every time, apart from fuzz.

### One period, high noise

In [ ]:
lambda_op_plot(
    results_sn1,
    file_stem="trig-sin-noise-01-complexity-mse-displot",
    mse_lims=(5e-3, 1.1e-2),
    complexity_lims=(0, 169),
    mse_bound=0.01,
    complexity_col="complexity_defuzz",
    spiffy_titles=[spiffy_Lopstr12, spiffy_Lopstr4, spiffy_Lopstr2],
    xlabel="Complexity (defuzzed)",
    ylabel="MSE",
)

In [ ]:
au.count_by_threshold(results_sn1, 0.9*sn1_mse_bound, groupby="Lopstr")

In [ ]:
results_sn1.loc[Lopstr12].sort_values(by="mse")

The `Lopstr12` runs are mostly wrong symbolically but accurate numerically.

In [ ]:
results_sn1.loc[Lopstr4].sort_values(by="mse")

Same for the `Lopstr4` runs.

In [ ]:
results_sn1.loc[Lopstr2].sort_values(by="mse")

In [ ]:
results_sn1.loc[Lopstr2].sympy_defuzz.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2))

The `Lopstr2` runs are mostly right apart from fuzz and cruft.

In [ ]:
results_sn1.loc[Lopstr2].sympy_defuzz.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-1))

These are generally correct, although you do have to be generous with the defuzzing to see it plainly.

So the story here is that if there's noise, the complexity penalty has to be of approximately the same order of magnitude as the noise, otherwise, it will add a lot of cruft trying to fit the noise.
The effect is very noticeable.
Which means that you have to estimate the noise before running symbolic regression.